In [1]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS
import scipy.stats as stats

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

2022-11-14 10:47:23 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Running on Apache Spark version 3.1.3
SparkUI available at http://compe073.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.102-817f6fb3468f
LOGGING: writing to logs/hail/hail_test_export.log


In [8]:
phenotypes = "data/phenotypes/spiros_brava_phenotypes_binary_200k.tsv"
input_path = "data/prs/hapmap/ukb_500k/fitting/ukb_hapmap_500k_eur_chr21"
input_type = "plink"

In [4]:
ht = hl.import_table(phenotypes,
             types={'eid': hl.tstr},
             missing=["",'""',"NA"],
             impute=True,
             force=True,
             key='eid'
             )

2022-11-14 10:47:54 Hail: INFO: Reading table to impute column types(0 + 1) / 1]
2022-11-14 10:48:37 Hail: INFO: Loading <StructExpression of type struct{eid: str, sex: int32, age: int32, `ukbb.centre`: int32, PC1: float64, PC2: float64, PC3: float64, PC4: float64, PC5: float64, PC6: float64, PC7: float64, PC8: float64, PC9: float64, PC10: float64, PC11: float64, PC12: float64, PC13: float64, PC14: float64, PC15: float64, PC16: float64, PC17: float64, PC18: float64, PC19: float64, PC20: float64, PC21: float64, PC22: float64, PC23: float64, PC24: float64, PC25: float64, PC26: float64, PC27: float64, PC28: float64, PC29: float64, PC30: float64, PC31: float64, PC32: float64, PC33: float64, PC34: float64, PC35: float64, PC36: float64, PC37: float64, PC38: float64, PC39: float64, PC40: float64, BC_combined: bool, BC_combined_primary_care: bool, CAD_combined: bool, CAD_combined_primary_care: bool, COPD_combined: bool, COPD_combined_primary_care: bool, CLD_combined: bool, CLD_combined_primary

NameError: name 'mt' is not defined

In [10]:
mt = io.import_table(input_path, input_type)
mt = mt.annotate_cols(pheno=ht[mt.s])

2022-11-14 10:49:32 Hail: INFO: Found 246271 samples in fam file.
2022-11-14 10:49:32 Hail: INFO: Found 16727 variants in bim file.


In [15]:
"CIRR_combined_primary_care" in list(mt.pheno)

True

In [17]:
defined = hl.is_defined(mt.pheno.CIRR_combined_primary_care).collect()

In [11]:
from ukb_utils import variants

In [3]:
p1 = "data/phased/wes_union_calls/200k/shapeit5/phase_rare/ukb_wes_union_calls_shapeit5_200k_chr21-20xshort/shapeit5_prs100000_pro25000_mprs150000.1of1.vcf.gz"
p2 = "data/prephased/wes_union_calls/ukb_wes_union_calls_200k_chr21.vcf.gz"

In [17]:
mt1 = io.import_table(p1, "vcf")
mt2 = io.import_table(p2, "vcf")

In [18]:
mt1 = mt1.annotate_entries(
    GT_rb = mt2[mt1.row_key, mt1.col_key].GT,
    PS_rb = mt2[mt1.row_key, mt1.col_key].PS
)

In [22]:
mt1.transmute_rows(
    info = mt1.info.annotate(
        AC_rb = mt2.rows()[mt1.row_key].info.AC,
        AN_rb = mt2.rows()[mt1.row_key].info.AN
    )
)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32, 
        AC_rb: int32, 
        AN_rb: int32
    }
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [16]:
mt1.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32
    }
    'AC_rb': int32
    'AN_rb': int32
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [20]:
mt1 = mt1.filter_rows(variants.get_mac_expr(mt1) > 1 )
mt2 = mt2.filter_rows(variants.get_mac_expr(mt2) > 1 )

In [21]:
print(mt1.count())
print(mt2.count())

(10700, 176618)


(10707, 185492)


In [22]:
mt1 = mt1.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt1.locus, mt1.alleles))
mt2 = mt2.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt2.locus, mt2.alleles))

In [23]:
print(mt1.aggregate_rows(hl.agg.sum(mt1.mismatch)))

1


In [24]:
print(mt2.aggregate_rows(hl.agg.sum(mt2.mismatch)))

0


In [ ]:
input_path="data/prs/hapmap/ukb_500k/fitting/ukb_hapmap_500k_eur_chrCHR"
input_type="plink"
phenotypes="/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/curated_phenotypes_cts.tsv"
phenotypes="Albumin_residual"
covariates=''
100
data/prs/sumstat/cts/ukb_hapmap_500k_eur_Albumin_residual_chrCHR


In [7]:
input_path = "data/mt/annotated/ukb_eur_wes_union_calls_200k_chr19.mt"

In [8]:
mt = io.import_table(input_path, "mt", calc_info = False)

In [22]:
mt.consequence.vep.worst_csq_for_variant_canonical.most_severe_consequence

<StringExpression of type str>

In [18]:
mt.consequence.vep.worst_csq_by_gene_canonical

<ArrayStructExpression of type array<struct{allele_num: int32, amino_acids: str, biotype: str, canonical: int32, ccds: str, cdna_start: int32, cdna_end: int32, cds_end: int32, cds_start: int32, codons: str, consequence_terms: array<str>, distance: int32, domains: array<struct{db: str, name: str}>, exon: str, gene_id: str, gene_pheno: int32, gene_symbol: str, gene_symbol_source: str, hgnc_id: str, hgvsc: str, hgvsp: str, hgvs_offset: int32, impact: str, intron: str, lof: str, lof_flags: str, lof_filter: str, lof_info: str, minimised: int32, polyphen_prediction: str, polyphen_score: float64, protein_end: int32, protein_start: int32, protein_id: str, sift_prediction: str, strand: int32, swissprot: str, transcript_id: str, trembl: str, uniparc: str, variant_allele: str, revel_score: float64, cadd_phred: float64, most_severe_consequence: str, csq_score: float64}>>

In [3]:
mt = io.import_table("data/unphased/wes_union_calls/bcftools/newtest/tagged.vcf.bgz", "vcf")

In [5]:
mt = mt.annotate_rows(
    missing = mt.aggregate_entries(hl.agg.sum(hl.is_defined(mt.GT)))
)

2022-11-04 15:54:49 Hail: INFO: scanning VCF for sortedness...
2022-11-04 16:00:11 Hail: INFO: Coerced sorted VCF - no additional import work to do


In [6]:
x = mt.missing.collect()

In [10]:
q = [27325787351-y for y in x]